# pMF Any-to-Any no Colab

Seleciona um runtime GPU T4. Configura os caminhos do clone com a extensão e do export YOLO Roboflow RF100 Corrosion Bi3Q3. O preflight corre apenas alguns updates de treino e gera exemplos de verificação. O treino de 50 épocas tem um comando separado na documentação.

In [2]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/andreferraria453/Synthetic-Data-Generation-Framework.git"
REPO = Path("/content/Synthetic-Data-Generation-Framework")
DATASET = Path("/content/corrosion-bi3q3")  # Ajusta à pasta que contém data.yaml
CACHE = Path("/content/pmf_any_to_any_features")
RUN = Path("/content/pmf_any_to_any_run")
BACKBONE = None

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
elif (REPO / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=REPO, check=True)

subprocess.run(["nvidia-smi", "-L"], check=True)

if not (REPO / "configs/pmf_any_to_any.json").is_file():
    raise FileNotFoundError("A extensão Any-to-Any ainda não está no GitHub.")

if not (DATASET / "data.yaml").is_file():
    raise FileNotFoundError(f"Ajusta DATASET: não encontrei data.yaml em {DATASET}")

print(f"Código: {REPO}\nDataset: {DATASET}")

FileNotFoundError: A extensão Any-to-Any ainda não está no GitHub.

## Dependências

Executa a instalação uma vez por runtime e reinicia o kernel se o Colab o pedir. Mantém o PyTorch CUDA do runtime. A T4 usa FP32 neste preflight.

In [ ]:
run("-m", "pip", "install", "-r", "requirements-any-to-any.txt")

## Auditoria, precompute curto e smoke test

O precompute usa duas imagens do treino e duas da validação. A primeira execução pode carregar checkpoints dos encoders e do pMF para a cache do Colab.

In [ ]:
run("-m", "unittest", "discover", "-s", "tests", "-v")
run("-m", "rae_experiment.any_to_any.data", "--dataset", DATASET, "--audit-only")
run("-m", "rae_experiment.any_to_any.data", "--dataset", DATASET,
    "--cache", CACHE, "--limit", "2")

In [ ]:
backbone_args = ["--backbone", BACKBONE] if BACKBONE else []
run("-m", "rae_experiment.any_to_any.train", "--config", "configs/pmf_any_to_any.json",
    "--dataset", DATASET, "--cache", CACHE, "--output", RUN, *backbone_args, "--smoke")

## Curvas e amostras

Define RUN com o diretório da experiência e volta a correr estas células à medida que forem sendo escritas épocas.

In [ ]:
import json
import matplotlib.pyplot as plt
from IPython.display import Image, display

rows = [json.loads(path.read_text()) for path in sorted((RUN / "epochs").glob("*.json"))]
if not rows:
    print("Ainda não existem épocas completas.")
for group in ("train", "validation", "metrics", "efficiency") if rows else ():
    series = [row[group] if group in row else {
        key: val for key, val in row.items() if key.endswith(("_seconds", "_mib"))
    } for row in rows]
    keys = sorted({key for values in series for key in values})
    for offset in range(0, len(keys), 4):
        figure, axes = plt.subplots(1, 4, figsize=(16, 3), constrained_layout=True)
        for axis in axes:
            axis.set_visible(False)
        for axis, key in zip(axes, keys[offset:offset + 4]):
            axis.set_visible(True)
            values = [(row["epoch"], entry[key]) for row, entry in zip(rows, series) if entry.get(key) is not None]
            if values:
                axis.plot(*zip(*values), marker=".")
            axis.set_title(f"{group}: {key}")
            axis.set_xlabel("Época")
        plt.show()

In [ ]:
EPOCH = None  # Inteiro para selecionar uma época específica
examples = sorted((RUN / "examples").glob("*/rgb_reference_generated_edit.png"))
if EPOCH is not None:
    examples = [path for path in examples if path.parent.name == f"{EPOCH:03}"]
if examples:
    print("Referência · geração livre · edição localizada")
    display(Image(filename=str(examples[-1])))